# Progress Review 1 — Verification
**Goal:** Confirm that transactions are flowing from Kafka → MinIO (Delta Lake).

Run each cell in order after starting the Kafka producer and Spark streaming job.

In [1]:
import os

MINIO_ENDPOINT   = 'http://minio:9000'
MINIO_ACCESS_KEY = 'admin'
MINIO_SECRET_KEY = 'bigdata123'

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('PR1-Verification')
    .master('local[2]')   # ← runs inside Jupyter, no version clash
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.1.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.eventLog.enabled', "false")
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark ready ✅')

Spark ready ✅


## 1. How many rows have landed in MinIO so far?

In [2]:
df = spark.read.format('delta').load('s3a://warehouse/raw/transactions')
total = df.count()
print(f'Total rows in Delta table: {total:,}')

Total rows in Delta table: 594,241


## 2. Preview the latest 10 records

In [3]:
from pyspark.sql.functions import desc
df.orderBy(desc('ingested_at')).select(
    'TransactionID', 'TransactionAmt', 'ProductCD',
    'isFraud', 'ingested_at'
).show(10, truncate=False)

+-------------+--------------+---------+-------+-----------------------+
|TransactionID|TransactionAmt|ProductCD|isFraud|ingested_at            |
+-------------+--------------+---------+-------+-----------------------+
|3393712      |481.7         |W        |0      |2026-04-27 08:35:43.371|
|3393927      |150.0         |H        |1      |2026-04-27 08:35:43.371|
|3393970      |36.614        |C        |1      |2026-04-27 08:35:43.371|
|3393822      |54.0          |W        |1      |2026-04-27 08:35:43.371|
|3393999      |36.614        |C        |1      |2026-04-27 08:35:43.371|
|3393987      |36.614        |C        |1      |2026-04-27 08:35:43.371|
|3393711      |21.751        |C        |0      |2026-04-27 08:35:43.371|
|3393752      |317.5         |W        |1      |2026-04-27 08:35:43.371|
|3393719      |59.0          |W        |0      |2026-04-27 08:35:43.371|
|3393844      |834.0         |W        |1      |2026-04-27 08:35:43.371|
+-------------+--------------+---------+-------+---

## 3. Fraud vs legitimate split

In [4]:
df.groupBy('isFraud').count().withColumnRenamed('count','num_transactions').show()

+-------+----------------+
|isFraud|num_transactions|
+-------+----------------+
|      1|           19908|
|      0|          579323|
+-------+----------------+



## 4. Delta Lake transaction history (proves ACID writes)

In [5]:
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, 's3a://warehouse/raw/transactions')
dt.history(5).select('version','timestamp','operation','operationMetrics').show(truncate=False)

+-------+-------------------+----------------+-----------------------------------------------------------------------------------------+
|version|timestamp          |operation       |operationMetrics                                                                         |
+-------+-------------------+----------------+-----------------------------------------------------------------------------------------+
|1302   |2026-04-27 08:36:34|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 534, numOutputBytes -> 50291, numAddedFiles -> 6}|
|1301   |2026-04-27 08:36:32|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 418, numOutputBytes -> 46364, numAddedFiles -> 6}|
|1300   |2026-04-27 08:36:28|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 278, numOutputBytes -> 40880, numAddedFiles -> 6}|
|1299   |2026-04-27 08:36:26|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 284, numOutputBytes -> 41788, numAddedFiles -> 6}|
|1298   |2026-04-27 08:36:24|STREAMING UP

## 5. Quick transaction amount stats
Shows the data is realistic (not zeros or nulls).

In [6]:
df.select('TransactionAmt').describe().show()

+-------+------------------+
|summary|    TransactionAmt|
+-------+------------------+
|  count|            603061|
|   mean|  133.079927456721|
| stddev|229.12381995834426|
|    min|             0.251|
|    max|         31937.391|
+-------+------------------+



In [ ]:
Confirmed and Verified! Thank you